# STAR-v3 — DANH GIA TREN OLD TEST SET (1,978 anh THAT + ground truth)

Day la **validation sim2real thuc su**: anh real-world + GT cong khai cua test set cu —
dung **protocol giong cac paper SOTA** (gallery 1,978, moi anh la GT cua dung 1 query) nen
**so sanh truc tiep duoc** voi so cong bo (~84.9/85.5/89.2 mAP — ho train tren data day du, ta moi 10K-hard).

**Danh gia 2 che do query** (do gap ngon ngu):
- `vlm`  : caption chi tiet (giong caption train cua ta)
- `human`: source_caption ngan kieu nguoi viet (gan voi paraphrase cua de thi nam nay)

**Cai dat:** GPU T4 · Internet ON · **Add Input 3 thu:**
1. Dataset `old-test` cua ban (attr.json + folder/zip `test/` 1,978 jpg)
2. Dataset cu (co `xvlm_16m_base.th`)
3. **Your Work → notebook train truoc do → OUTPUT** (chua cac `best.pth`) — notebook nay tu tim
   MOI best.pth trong input va danh gia het.

⚠️ **Pose caveat:** cac checkpoint hien tai train voi pose ON nhung old-test CHUA co ViTPose keypoints
→ eval khong fuse pose (lech nhe so voi luc train; so sanh TUONG DOI giua cac checkpoint van cong bang,
so tuyet doi bi danh gia THAP hon thuc luc). Neu team data trich ViTPose cho 1,978 anh nay (file
`*vitpose*.json` cung format cu, upload vao dataset old-test) → notebook tu dung.

**Thoi gian:** ~10' setup + ~5'/checkpoint (2 che do).

In [ ]:
# [1/6] GPU + clone/pull repo
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Bat GPU T4 trong Settings!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
print("repo OK:", os.getcwd())

In [ ]:
# [2/6] Env pinned + X-VLM + patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/6] Tim inputs: old-test (attr.json + anh), X-VLM base ckpt, MOI best.pth da train
import glob, pathlib

def find_all(pattern):
    return sorted(set(glob.glob(f"/kaggle/input/*/{pattern}") +
                      glob.glob(f"/kaggle/input/*/**/{pattern}", recursive=True)))

attr = find_all("attr.json")
assert attr, "Khong thay attr.json — Add Input dataset old-test!"
ATTR = attr[0]
OLD_ROOT = str(pathlib.Path(ATTR).parent)
# anh co the nam o OLD_ROOT/test/0.jpg (giu cay) — kiem chung
probe = find_all("test/0.jpg") or find_all("0.jpg")
assert probe, "Khong thay anh test/0.jpg trong dataset old-test"
if not pathlib.Path(OLD_ROOT, "test/0.jpg").exists():
    OLD_ROOT = str(pathlib.Path(probe[0]).parents[1])   # .../<root>/test/0.jpg

base = find_all("xvlm_16m_base.th")
assert base, "Khong thay xvlm_16m_base.th — Add Input dataset cu!"
BASE_CKPT = base[0]

CKPTS = find_all("best.pth")
assert CKPTS, "Khong thay best.pth nao — Add Input: Your Work -> notebook train -> Output!"
VITPOSE = (find_all("*vitpose*.json") or [None])[0]
print("OLD_ROOT  =", OLD_ROOT)
print("BASE_CKPT =", BASE_CKPT)
print("checkpoints:", *CKPTS, sep="\n  ")
print("vitpose old-test:", VITPOSE or "(khong co -> eval khong fuse pose)")

In [ ]:
# [4/6] Build 2 manifest tu attr.json (da kiem tren data that: 1,978 rows, du file)
# + TUY CHON: them distractor synthetic tu tar.zst neu attach (mac dinh TAT — xem ghi chu cuoi)
import json
import pandas as pd

N_SYNTH_DISTRACTORS = 0      # >0 (vd 4000): them anh synthetic lam distractor (can attach tar.zst)

rows = [json.loads(l) for l in open(ATTR, encoding="utf-8")]
pose_items = {}
if VITPOSE:
    pose_items = json.load(open(VITPOSE, encoding="utf-8")).get("items", {})

def kpts_of(key):
    it = pose_items.get(str(key))
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None

def build(mode):
    out = []
    for r in rows:
        cap = (r["caption"][0] if mode == "vlm" else (r.get("source_caption") or r["caption"])[0])
        out.append(dict(image_path=r["image"], caption=cap, split="valb",
                        sequence_id=f'old{r["image_id"]}', scene=f'old{r["image_id"]}',
                        action=str(r.get("normal", "unk")), image_id=str(r["image_id"]),
                        bbox=None, keypoints=kpts_of(r["image_id"])))
    df = pd.DataFrame(out)
    if N_SYNTH_DISTRACTORS > 0:   # distractor synthetic (yeu hon distractor that — xem ghi chu)
        import glob as g
        webp = g.glob("/kaggle/working/extract/**/train_webp/**/*.webp", recursive=True)[:N_SYNTH_DISTRACTORS]
        assert webp, "N_SYNTH_DISTRACTORS>0 nhung chua giai nen tar.zst (xem notebook train cell 3)"
        dd = pd.DataFrame([dict(image_path=p, caption="", split="valb", sequence_id=f"sd{i}",
                                scene=f"sd{i}", action="distractor", image_id=f"sd{i}",
                                bbox=None, keypoints=None) for i, p in enumerate(webp)])
        df = pd.concat([df, dd], ignore_index=True)
    return df

MANIFESTS = {}
for mode in ["vlm", "human"]:
    m = f"/kaggle/working/oldtest_{mode}.parquet"
    build(mode).to_parquet(m, index=False)
    MANIFESTS[mode] = m
kc = pd.read_parquet(MANIFESTS["vlm"]).keypoints.notna().mean()
print(f"manifests OK: {list(MANIFESTS)} | 1,978 query | kpts coverage: {kc:.0%}")

In [ ]:
# [5/6] EVAL: moi checkpoint x 2 che do query (inline — build model 1 lan/ckpt)
import sys, time, torch
sys.path.insert(0, "src")
from star.config import load_config, _merge
from star.data import PABDataset
from star.engine import evaluate_retrieval
from star.models import STARModel

RESULTS = []
for ck in CKPTS:
    name = pathlib.Path(ck).parent.name or pathlib.Path(ck).stem
    print("=" * 90); print(f">>> {name}  ({ck})")
    raw = torch.load(ck, map_location="cpu", weights_only=False)
    cfg = load_config("configs/star_v3_10k_kaggle.yaml")
    emb = (raw.get("extra") or {}).get("cfg") or {}
    if "model" in emb:
        _merge(cfg.model, emb["model"])
    cfg.model.checkpoint = BASE_CKPT
    model = STARModel(cfg).to("cuda").eval()
    msg = model.load_state_dict(raw["model"], strict=False)
    print(f"loaded: missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)} "
          f"| pose_enabled={cfg.model.pose_enabled} | val-mAP luc train={raw.get('best_metric'):.4f}")
    del raw
    for mode, mpath in MANIFESTS.items():
        ds = PABDataset(mpath, OLD_ROOT, model.backbone.tokenizer, split="valb",
                        image_size=cfg.data.image_size, max_token=cfg.data.max_token, train=False)
        t0 = time.time()
        rep = evaluate_retrieval(model, ds, "cuda", batch_size=64, num_workers=2)
        rep = {k: round(v, 4) for k, v in rep.items()}
        print(f"  [{mode:5s}] {rep}  ({(time.time()-t0)/60:.1f} phut)")
        RESULTS.append(dict(ckpt=name, mode=mode, **rep))
    del model
    torch.cuda.empty_cache()

In [ ]:
# [6/6] BANG TONG HOP
import pandas as pd
tab = pd.DataFrame(RESULTS).sort_values(["mode", "mAP"], ascending=[True, False])
print(tab.to_string(index=False))
print("\nMoc tham chieu (CUNG protocol gallery ~1,978 real, ho train data DAY DU):")
print("  SOTA cong bo ~ 84.9 / 85.5 / 89.2 mAP  |  VAL-B synthetic cua ta ~ 0.66")
print("Doc: gap 'vlm' vs 'human' = do lech ngon ngu query; gap voi SOTA = data 10K vs full + sim2real + (pose khong fuse)")

## Ve cau hoi DISTRACTOR (quan trong)
Old-test **khong co distractor rieng** — nhung gallery 1,978 anh (moi anh la GT cua 1 query, 1,977 anh
con lai la 'doi thu' tu nhien) chinh la **protocol cua cac paper SOTA** → so sanh duoc voi so cong bo. Do la gia tri chinh.

Muon them distractor:
1. ✅ **Hop le nhung YEU**: anh synthetic tu PAB train (`N_SYNTH_DISTRACTORS` o cell 4 + attach tar.zst).
   Yeu vi khac domain (synthetic vs real) — model phan biet duoc ngay → khong phan anh do kho that cua 36K gallery real.
2. ✅ Hop le va TOT hon: anh nguoi that tu dataset public ngoai (CUHK-PEDES/ICFG/MSMT17...) — can tai them, de sau.
3. ❌ **CAM**: dung gallery masked cua test NAM NAY lam distractor — luat cam dung test cho validation.

## Y nghia cua 2 che do query
- `vlm` cao, `human` thap nhieu → **gap ngon ngu** la van de → uu tien paraphrase-aug khi train.
- Ca hai cung thap so voi VAL-B 0.66 → **gap sim2real** dang ke hon du doan → can xem lai (vd domain-align ablation).
- Day la lan dau ta do duoc 2 gap nay TACH BACH — gia tri lon nhat cua old-test.